In [1]:
# GBD mortality indicators: 5q0, 28q0 and the derived 5q28 ####################
# Three GBD life-table quantities on one axis, for the same locations the
# neonatal residual-step pdfs cover (2026_10_09_neonatal.ipynb, cell 2):
#   global_superregions - fhs_hierarchy levels 0 and 1   (8 locations)
#   countries           - fhs_hierarchy level 3        (204 locations)
# Nine locations per page rather than one, so these are small multiples.
#
# All three are qx (life_table_parameter_id = 3), probabilities, from
# input/gbd_prevalence. No aggregation is done here: GBD's own files already
# carry Global, the super regions and every country, so the aggregates are
# GBD's rather than a population-weighted reconstruction. Checked: all 212
# locations x 2 sexes x 75 years are present in each of the three files with
# no gaps.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

Q_ROOT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition"
Q_GBD = f"{Q_ROOT}/input/gbd_prevalence"
Q_HIER = f"{Q_ROOT}/input/fhs_hierarchy.parquet"
Q_OUT = f"{Q_ROOT}/child_mortality/plots"
Q_PER_PAGE = 9  # 3 x 3

# Series order is FIXED, not sorted by value: slots 1-3 of the reference
# categorical palette, which are the three that validate on the all-pairs
# list (worst pair CVD dE 9.2, normal-vision 24.0 on the light surface).
# A fourth indicator here would need a facet, not a fourth hue.
Q_SERIES = [
    ("5q0", "gbd_mean_child_mortality_prevalence.parquet", "#2a78d6"),
    ("28q0", "gbd_mean_neonatal_mortality_prevalence.parquet", "#eb6834"),
    ("5q28", "gbd_mean_5q28_prevalence.parquet", "#1baf7a"),
]
Q_SEX = [(1, "-", "Male"), (2, "--", "Female")]  # sex is the secondary encoding
Q_INK = "#0b0b0b"
Q_INK2 = "#52514e"
Q_GRID = "#e1e0d9"

q_hier = pd.read_parquet(Q_HIER)
q_name = q_hier.set_index("location_id")["location_name"]

q_df = None
for label, fname, _colour in Q_SERIES:
    d = (
        pd.read_parquet(f"{Q_GBD}/{fname}")
        .reset_index()
        .drop(columns="age_group_id")
        .rename(columns={"gbd_mean_prevalence": label})
    )
    keys = ["location_id", "year_id", "sex_id"]
    q_df = d if q_df is None else q_df.merge(d, on=keys, how="outer")

# The identity the derived series rests on, re-checked on what is about to be
# drawn rather than taken on trust: 1 - 5q0 = (1 - 28q0)(1 - 5q28).
_lhs = 1 - q_df["5q0"]
_rhs = (1 - q_df["28q0"]) * (1 - q_df["5q28"])
assert np.allclose(_lhs, _rhs), "5q28 does not reconstruct 5q0"
assert q_df[["5q0", "28q0", "5q28"]].notna().all().all(), "missing values"

Q_SETS = {
    "global_superregions": sorted(q_hier[q_hier.level.isin([0, 1])].location_id),
    "countries": sorted(q_hier[q_hier.level == 3].location_id),
}
for _tag, _locs in Q_SETS.items():
    assert set(_locs) <= set(q_df.location_id), f"{_tag}: locations missing from GBD"

print(
    f"{len(q_df):,} rows, {q_df.location_id.nunique()} locations, "
    f"years {q_df.year_id.min()}-{q_df.year_id.max()}"
)
for _tag, _locs in Q_SETS.items():
    print(
        f"  {_tag:20s} {len(_locs):>3} locations -> "
        f"{-(-len(_locs) // Q_PER_PAGE)} pages at {Q_PER_PAGE}/page"
    )

138,750 rows, 925 locations, years 1950-2024
  global_superregions    8 locations -> 1 pages at 9/page
  countries            204 locations -> 23 pages at 9/page


In [2]:
# Nine locations per page #####################################################
# Each panel is one location: three indicators by colour, two sexes by dash.
# Both sexes are kept rather than averaged - these are per-birth probabilities,
# so a plain mean of the two would need births to weight it.
#
# Three choices worth stating:
#   * y starts at 0 and is scaled PER PANEL. The question these pages answer is
#     how the three indicators sit against each other within a location, and a
#     shared scale would flatten every low-mortality panel on a page that also
#     holds a high-mortality one. Compare within a panel, not across them.
#   * every panel carries direct labels at the right edge. Aqua (slot 3) sits
#     below 3:1 against a white surface, so the palette's relief rule applies:
#     identity cannot rest on colour alone, and a pdf has no hover or table
#     view to fall back on. Label text is ink; the coloured dot beside it
#     carries the identity.
#   * those labels are de-collided. 28q0 and 5q28 converge wherever neonatal
#     deaths come to dominate under-5 deaths - which is most places by 2024 -
#     and printed at their true heights the two strings overlap into mush. Each
#     label is pushed to its own slot and joined to its dot by a leader.

Q_LABEL_GAP = 0.058  # minimum vertical separation, in axes fraction


def _q_label_slots(values, gap=Q_LABEL_GAP):
    """Nudge label positions apart, preserving their order.

    values: list of (key, y_fraction). Returns {key: y_fraction}, spread so no
    two sit closer than `gap`, then shifted back inside [0, 1] if the stack ran
    off the top.
    """
    ordered = sorted(values, key=lambda kv: kv[1])
    out, prev = [], None
    for key, y in ordered:
        y = y if prev is None else max(y, prev + gap)
        out.append((key, y))
        prev = y
    overflow = out[-1][1] - 1.0
    if overflow > 0:
        out = [(k, y - overflow) for k, y in out]
    return dict(out)


def q_page_panel(ax, loc_id, *, label_series):
    blk = q_df[q_df.location_id == loc_id]
    ymax = 0.0
    ends = {}
    for label, _fname, colour in Q_SERIES:
        for sex_id, ls, _sex_label in Q_SEX:
            s = blk[blk.sex_id == sex_id].sort_values("year_id")
            ax.plot(
                s.year_id, s[label], color=colour, ls=ls, lw=1.6, solid_capstyle="round"
            )
            ymax = max(ymax, float(s[label].max()))
        s = blk[blk.sex_id == 1].sort_values("year_id")  # anchor on the solid line
        ends[label] = (float(s.year_id.iloc[-1]), float(s[label].iloc[-1]), colour)

    top = ymax * 1.08 if ymax > 0 else 1.0
    ax.set_ylim(0, top)
    ax.set_xlim(int(q_df.year_id.min()), int(q_df.year_id.max()))
    ax.set_xticks([1950, 1975, 2000, 2024])

    if label_series:
        slots = _q_label_slots([(k, v[1] / top) for k, v in ends.items()])
        for label, (x, y, colour) in ends.items():
            ax.plot([x], [y], marker="o", ms=3.2, color=colour, zorder=4, clip_on=False)
            ys = slots[label]
            # leader from the true value out to the label's own slot
            ax.annotate(
                label,
                xy=(1.0, y / top),
                xycoords="axes fraction",
                xytext=(1.035, ys),
                textcoords="axes fraction",
                va="center",
                ha="left",
                fontsize=7,
                color=Q_INK2,
                annotation_clip=False,
                arrowprops=dict(
                    arrowstyle="-", color=colour, lw=0.6, shrinkA=1.0, shrinkB=1.5
                ),
            )

    ax.grid(True, color=Q_GRID, lw=0.6, alpha=0.9)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(Q_GRID)
    ax.tick_params(labelsize=7, colors=Q_INK2, length=3)
    ax.set_title(
        f"{q_name.get(loc_id, loc_id)}", fontsize=8.5, color=Q_INK, loc="left", pad=4
    )


def q_make_pdf(loc_ids, out_path, note):
    pages = [loc_ids[i : i + Q_PER_PAGE] for i in range(0, len(loc_ids), Q_PER_PAGE)]
    with PdfPages(out_path) as pdf:
        for page_no, chunk in enumerate(pages, 1):
            fig, axes = plt.subplots(3, 3, figsize=(13, 9), squeeze=False)
            flat = axes.ravel()
            for ax, loc_id in zip(flat, chunk):
                q_page_panel(ax, loc_id, label_series=True)
            for ax in flat[len(chunk) :]:
                ax.axis("off")
            handles = [
                plt.Line2D([], [], color=c, lw=1.6, label=lab)
                for lab, _f, c in Q_SERIES
            ] + [
                plt.Line2D([], [], color=Q_INK2, lw=1.6, ls=ls, label=sex_label)
                for _s, ls, sex_label in Q_SEX
            ]
            fig.legend(
                handles=handles,
                loc="lower center",
                ncol=5,
                frameon=False,
                fontsize=9,
                bbox_to_anchor=(0.5, 0.005),
            )
            fig.suptitle(
                "GBD probability of death: 5q0, 28q0 and 5q28"
                f"   -   {note}   -   page {page_no} of {len(pages)}",
                fontsize=12,
                color=Q_INK,
                x=0.012,
                ha="left",
                y=0.985,
            )
            fig.supylabel("Probability of death", fontsize=9, color=Q_INK2)
            fig.text(
                0.012,
                0.955,
                "5q28 = 1 - (1 - 5q0) / (1 - 28q0), the probability of "
                "dying before age 5 given survival to day 28. "
                "y starts at 0 and is scaled per panel.",
                fontsize=8,
                color=Q_INK2,
                ha="left",
            )
            fig.tight_layout(rect=[0.015, 0.045, 1, 0.945], w_pad=3.2)
            pdf.savefig(fig)
            plt.close(fig)
    print(f"Saved {out_path}  ({len(pages)} pages, {len(loc_ids)} locations, {note})")


Q_NOTES = {
    "global_superregions": "Global and the super regions",
    "countries": "level-3 countries",
}
for _tag, _locs in Q_SETS.items():
    q_make_pdf(_locs, f"{Q_OUT}/gbd_5q0_28q0_5q28_{_tag}.pdf", Q_NOTES[_tag])

Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/gbd_5q0_28q0_5q28_global_superregions.pdf  (1 pages, 8 locations, Global and the super regions)
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/gbd_5q0_28q0_5q28_countries.pdf  (23 pages, 204 locations, level-3 countries)
